Simulacion de cola de espera de K servidores

In [20]:
import numpy as np
import pandas as pd
import datetime

funciones auxiliares

In [21]:
def acumular_array(array):
    acumulado = 0
    array_acumulado = []
    for n in array :
        acumulado = n + acumulado
        array_acumulado.append(acumulado)
    return array_acumulado


1.establecemos los valores iniciales

In [22]:
servidores_activos = 1
llegadas = 10 #por hora
capacidad_servicio = 12 #por hora y por servidor
hora_inicio = datetime.datetime.strptime('09:00', '%H:%M')
hora_final = datetime.datetime.strptime('14:00', '%H:%M')

2.creacion de clientes

In [23]:
df = pd.DataFrame()

#generacion de tiempos de llegadas
tiempo_entre_llegadas = []

horas = []
aux_time=hora_inicio
while(aux_time < hora_final):
    aux_llegada = (-np.log(1-np.random.uniform(0,1))/llegadas)*60
    tiempo_entre_llegadas.append(aux_llegada)
    aux_time = aux_time + datetime.timedelta(minutes=aux_llegada)
    horas.append(aux_time.strftime("%H:%M:%S"))
momento_de_llegada = acumular_array(tiempo_entre_llegadas)

df.insert(0 , 'hora de llegada' , horas)
df.insert(1 , 'tiempo entre llegadas', tiempo_entre_llegadas)
df.insert(2 , 'momento de llegada' , momento_de_llegada)
df

,hora de llegada,tiempo entre llegadas,momento de llegada
0,09:04:14,4.242672,4.242672
1,09:06:19,2.090496,6.333167
2,09:16:32,10.209013,16.542180
3,09:17:39,1.123345,17.665525
4,09:27:24,9.739671,27.405195
5,09:36:36,9.201099,36.606294
6,09:38:06,1.504584,38.110878
7,09:45:52,7.764241,45.875119
8,09:57:07,11.255617,57.130736
9,10:02:55,5.789135,62.919871


3.creacion de servidores y atencion

In [24]:
servidores = [0.0 for _ in range(servidores_activos)]
num_clientes = len(momento_de_llegada)

tiempo_inicio_servicio = np.zeros(num_clientes)
tiempo_terminacion_servicio = np.zeros(num_clientes)
tiempo_de_servicio = [(-np.log(1-np.random.uniform(0,1))/capacidad_servicio)*60 for _ in range(num_clientes)]
cliente_servidor = np.zeros(num_clientes)

for i in range(num_clientes):
        mejor_servidor = servidores.index(min(servidores))
        tiempo_inicio_servicio[i] = max(momento_de_llegada[i], servidores[mejor_servidor])
        tiempo_terminacion_servicio[i] = tiempo_inicio_servicio[i] + tiempo_de_servicio[i]
        servidores[mejor_servidor] = tiempo_terminacion_servicio[i]
        cliente_servidor[i] = mejor_servidor+1 

tiempo_espera = [ inicio_servicio - llegada for inicio_servicio,llegada in zip(tiempo_inicio_servicio,momento_de_llegada)]

df.insert(3 , 'tiempo de espera', tiempo_espera)
df.insert(4 , 'tiempo de inicio de servicio', tiempo_inicio_servicio)
df.insert(5 , 'tiempo de servicio', tiempo_de_servicio)
df.insert(6 , 'tiempo de terminacion de servicio', tiempo_terminacion_servicio)
df.insert(7 , 'servidor', cliente_servidor)
df.to_csv('cola.csv',index=False)
df

,hora de llegada,tiempo entre llegadas,momento de llegada,tiempo de espera,tiempo de inicio de servicio,tiempo de servicio,tiempo de terminacion de servicio,servidor
0,09:04:14,4.242672,4.242672,0.000000,4.242672,0.837482,5.080154,1.0
1,09:06:19,2.090496,6.333167,0.000000,6.333167,5.910253,12.243421,1.0
2,09:16:32,10.209013,16.542180,0.000000,16.542180,12.654393,29.196573,1.0
3,09:17:39,1.123345,17.665525,11.531048,29.196573,3.954644,33.151217,1.0
4,09:27:24,9.739671,27.405195,5.746021,33.151217,12.625892,45.777108,1.0
5,09:36:36,9.201099,36.606294,9.170814,45.777108,2.124284,47.901393,1.0
6,09:38:06,1.504584,38.110878,9.790515,47.901393,1.031883,48.933276,1.0
7,09:45:52,7.764241,45.875119,3.058156,48.933276,3.640352,52.573628,1.0
8,09:57:07,11.255617,57.130736,0.000000,57.130736,4.079390,61.210126,1.0
9,10:02:55,5.789135,62.919871,0.000000,62.919871,5.116764,68.036635,1.0


In [25]:
print(f"promedio de tiempo entre llegadas: {sum(tiempo_entre_llegadas)/len(tiempo_entre_llegadas)}")
print(f"promedio de tiempo de espera: {sum(tiempo_espera)/len(tiempo_espera)}")
print(f"promedio de tiempo de servicio: {sum(tiempo_de_servicio)/len(tiempo_de_servicio)}")


promedio de tiempo entre llegadas: 6.170593508890049
promedio de tiempo de espera: 4.92685704228637
promedio de tiempo de servicio: 4.686714459718395
